# 03 · Extracción de jugadas y construcción del dataset (Paso 4B)
Implementación de `docs/action-extraction-spec.md`. Consume los manifiestos de los notebooks 01 y 02, sin recalibrar la interfaz.

**Orden:** configuración → funciones → carga → piloto temporal → catálogo → galería espacial → lote.
Este notebook es autocontenido: no requiere importar módulos nuevos del repositorio.

Dependencias (si faltan, ejecutar en una celda):
```python
%pip install "numpy>=1.26,<3" "opencv-python-headless>=4.10,<6" "matplotlib>=3.8,<4" "requests>=2.31,<3" "boto3>=1.34,<2"
```
No contiene credenciales. Usa variables de entorno o Kaggle Secrets como los notebooks anteriores.


In [ ]:
import os, json, math, hashlib, tempfile, time
from pathlib import Path
from dataclasses import dataclass, asdict
import cv2
import numpy as np
import requests
import matplotlib.pyplot as plt


## Configuración y control de R2
`UPLOAD_TO_R2 = False` bloquea **todas** las subidas, incluidas imágenes y metadatos, aunque existan credenciales. `READ_FROM_R2` es independiente: se puede leer sin escribir.

Para trabajar completamente con entradas locales: `READ_FROM_R2=False`, colocar `segmented_matches.json` y `ui_layouts.json` en `WORK_DIR/manifests/`, y videos en `WORK_DIR/videos/`. El catálogo se descarga una vez y queda en caché; también se admite un catálogo local.

Las cuatro fases están disponibles. `RUN_BATCH=False` permite empezar revisando el piloto; activarlo para procesar la selección completa. Las seis etiquetas se conservan exactamente como esquema preliminar del documento.


In [ ]:
# ÚNICO INTERRUPTOR DE ESCRITURA REMOTA. False permite leer R2, pero jamás subir.
UPLOAD_TO_R2 = False
READ_FROM_R2 = True
VIDEO_FILTER = 'all'  # 'all', un número o una lista de IDs
PILOT_VIDEO_ID = None  # None selecciona el primer video con segmentación y layout
PILOT_MATCH_INDEX = 0  # Índice cero dentro de la lista de partidas
RUN_BATCH = False  # Activar después de revisar las fases 1–3
SKIP_COMPLETED = True
DELETE_DOWNLOADED_VIDEOS = True
WORK_DIR = Path('/kaggle/working') if Path('/kaggle').exists() else Path('data')
OUTPUT_DIR = WORK_DIR / 'action-extraction'
VIDEOS_DIR = WORK_DIR / 'videos'
MANIFESTS_DIR = WORK_DIR / 'manifests'
CATALOG_DIR = OUTPUT_DIR / 'catalog'
CDN_URL = os.getenv('CDN_URL', 'https://cdn-cr.eddux.dev').rstrip('/')
CATALOG_URL = 'https://raw.githubusercontent.com/RoyaleAPI/cr-api-data/master/docs/json/cards_i18n_evo.json'
ICON_URL = 'https://raw.githubusercontent.com/RoyaleAPI/cr-api-assets/master/cards/{key}.png'
# Opcional: JSON local con registros {key, name_es, elixir, image_path, is_evolved}.
# Sirve también para añadir recortes de referencia verificados de los videos.
CUSTOM_CATALOG = None

@dataclass(frozen=True)
class ExtractionConfig:
    scan_fps: float = 10.0
    min_elixir_drop: float = 1.5
    drop_window: float = 0.5
    pre_decision: float = 0.65
    card_lookback: float = 0.8
    rotation_threshold: float = 0.20
    card_score_min: float = 0.48
    card_margin_min: float = 0.025
    cost_tolerance: float = 0.85
    wait_fraction: float = 0.175
    wait_stability: float = 3.0
    seed: int = 42
    # ROI de relleno dentro de elixir_bar_box; excluye gota/número y Máx:10.
    # Ajustar en la gráfica piloto si la interfaz difiere; la caja del NB02 es completa.
    elixir_fill_roi: tuple = (0.23, 0.57, 0.76, 0.20)

CFG = ExtractionConfig()
assert 0.5 <= CFG.pre_decision <= 0.8
assert 0.15 <= CFG.wait_fraction <= 0.20
for directory in (OUTPUT_DIR, VIDEOS_DIR, MANIFESTS_DIR, CATALOG_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print('Subidas a R2:', 'ACTIVADAS' if UPLOAD_TO_R2 else 'DESACTIVADAS — salida local')


## Utilidades de entrada y persistencia local
Cajas `[x,y,ancho,alto]`, reescaladas a la resolución del video. Imágenes en BGR internamente y RGB al mostrarlas.

In [ ]:
def atomic_bytes(path, content):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(prefix=path.name + '.', dir=path.parent)
    try:
        with os.fdopen(fd, 'wb') as f:
            f.write(content)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp, path)
    finally:
        if os.path.exists(tmp):
            os.unlink(tmp)


def atomic_json(path, value):
    atomic_bytes(path, json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False).encode())


def get_config(key, default=None):
    value = os.getenv(key)
    if value is not None:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key)
    except Exception:
        return default


def make_s3():
    import boto3
    from botocore.config import Config
    keys = [get_config(k) for k in ('R2_ENDPOINT', 'R2_ACCESS_KEY', 'R2_SECRET_KEY')]
    if not all(keys):
        return None
    return boto3.client('s3', endpoint_url=keys[0], aws_access_key_id=keys[1],
                        aws_secret_access_key=keys[2], config=Config(signature_version='s3v4'))


def read_manifest(name, client=None):
    # Lectura S3 directa: evita caché CDN. No confundir lectura con permiso de subida.
    local = MANIFESTS_DIR / name
    if READ_FROM_R2 and client is not None:
        body = client.get_object(Bucket=R2_BUCKET, Key='metadata/' + name)['Body'].read()
        value = json.loads(body)
        atomic_json(local, value)
        return value
    if not local.exists():
        raise FileNotFoundError(f'Falta {local}. Copia el manifiesto del notebook anterior o configura lectura R2.')
    return json.loads(local.read_text())


def fetch_video(video_id):
    if not video_id or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-' for c in video_id):
        raise ValueError('video_id no válido')
    path = VIDEOS_DIR / f'{video_id}.mp4'
    if path.exists():
        return path, False
    partial = path.with_suffix('.mp4.part')
    try:
        with requests.get(f'{CDN_URL}/videos/{video_id}.mp4', stream=True, timeout=(20, 120)) as response:
            response.raise_for_status()
            with partial.open('wb') as f:
                for chunk in response.iter_content(1024 * 1024):
                    f.write(chunk)
        cap = cv2.VideoCapture(str(partial))
        ok = cap.isOpened() and cap.get(cv2.CAP_PROP_FRAME_COUNT) > 0
        cap.release()
        if not ok:
            raise ValueError('La descarga no contiene un video decodificable')
        os.replace(partial, path)
    finally:
        partial.unlink(missing_ok=True)
    return path, True


class VideoReader:
    def __init__(self, path):
        self.cap = cv2.VideoCapture(str(path))
        self.fps = self.cap.get(cv2.CAP_PROP_FPS)
        self.count = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if not self.cap.isOpened() or self.fps <= 0 or not self.count:
            self.cap.release()
            raise ValueError(f'Video ilegible: {path}')

    def at(self, sec):
        idx = int(math.floor(sec * self.fps + 1e-6))
        if not 0 <= idx < self.count:
            raise ValueError(f'Frame fuera del video: {sec}')
        current = int(self.cap.get(cv2.CAP_PROP_POS_FRAMES))
        if current <= idx and idx-current <= self.fps:
            for _ in range(idx-current):
                if not self.cap.grab():
                    raise IOError(f'No se pudo avanzar a t={sec}')
        else:
            self.cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = self.cap.read()
        if not ok:
            raise IOError(f'No se pudo decodificar t={sec}')
        return frame

    def close(self):
        self.cap.release()


def boxes_for(layout, frame):
    h, w = frame.shape[:2]
    rw, rh = layout['resolution']
    def scaled(box):
        x, y, bw, bh = box
        x0, y0 = round(x*w/rw), round(y*h/rh)
        x1, y1 = round((x+bw)*w/rw), round((y+bh)*h/rh)
        if not (0 <= x0 < x1 <= w and 0 <= y0 < y1 <= h):
            raise ValueError(f'Caja inválida en layout: {box}')
        return [x0, y0, x1-x0, y1-y0]
    px = layout['boxes_pixel']
    return {'arena_box': scaled(px['arena_box']), 'elixir_bar_box': scaled(px['elixir_bar_box']),
            'card_slots': [scaled(b) for b in px['card_slots']]}


def crop(frame, box):
    x, y, w, h = map(int, box)
    return frame[y:y+h, x:x+w]


def slot_art(slot):
    h, w = slot.shape[:2]
    # Excluye borde, coste inferior y margen superior de elevación.
    return slot[int(.23*h):int(.77*h), int(.15*w):int(.85*w)]


def slot_signature(slot):
    # Al gastar elixir, otras cartas pasan a gris por falta de recursos.
    # Usar estructura en gris evita confundir ese cambio global con rotaciones.
    gray = cv2.cvtColor(cv2.resize(slot_art(slot), (32, 40)), cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (3,3), 0).astype(np.float32)
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1)
    magnitude = cv2.magnitude(gx, gy)
    return cv2.normalize(magnitude, None).flatten().astype(np.float32)


def signature_distance(a, b):
    return float(cv2.compareHist(a, b, cv2.HISTCMP_BHATTACHARYYA))


def chat_occluded(slots):
    # La firma azul debe cubrir los cuatro slots, no solo una carta azul.
    signatures = []
    for slot in slots:
        hsv = cv2.cvtColor(slot_art(slot), cv2.COLOR_BGR2HSV)
        blue = ((hsv[:,:,0] >= 90) & (hsv[:,:,0] <= 125) & (hsv[:,:,1] > 85))
        signatures.append(float(blue.mean()))
    return min(signatures) > 0.68


def read_elixir(frame, box, cfg=CFG):
    roi = crop(frame, box)
    h, w = roi.shape[:2]
    x, y, rw, rh = cfg.elixir_fill_roi
    strip = roi[round(y*h):max(round((y+rh)*h), round(y*h)+1), round(x*w):round((x+rw)*w)]
    rgb = cv2.cvtColor(strip, cv2.COLOR_BGR2RGB).astype(np.int16)
    mask = (rgb[:,:,0] > 165) & (rgb[:,:,2] > 155) & (rgb[:,:,1] < 110) & (rgb[:,:,0]-rgb[:,:,1] > 60)
    # Cierre horizontal de separadores de celdas; no cuenta gota ni coste.
    columns = (mask.mean(axis=0) >= 0.25).astype(np.uint8)[None, :]
    columns = cv2.morphologyEx(columns, cv2.MORPH_CLOSE, np.ones((1, max(3, strip.shape[1]//80)), np.uint8))[0]
    xs = np.flatnonzero(columns)
    if not len(xs):
        return 0.0
    if xs[0] > .12*len(columns):
        return float('nan')
    return float(np.clip(10 * (xs[-1]+1)/len(columns), 0, 10))


## Subsistema 1 · Detector temporal
Muestreo a 10 Hz para resolver caídas en ≤0.5 s. Exige caída neta ≥1.5 y cambio persistente de slot. Pausa durante chat azul en los cuatro slots. Une escalones de la misma caída; no impone un bloqueo de 1.5 s, de modo que dos gastos separados pueden formar un combo.

Dos rotaciones dentro de una caída sin separación observable se registran como ambiguas. Las cartas de coste 1 quedan fuera del umbral principal solicitado; no se cambia silenciosamente esta regla.


In [ ]:
def scan_match(reader, match, layout, cfg=CFG):
    start, end = float(match['start_sec']), float(match['end_sec'])
    boxes = boxes_for(layout, reader.at(start))
    times = np.arange(start, end + 1e-6, 1/cfg.scan_fps)
    elixir, signatures, occluded, changes = [], [], [], []
    for i, t in enumerate(times):
        frame = reader.at(float(t))
        slots = [crop(frame, b) for b in boxes['card_slots']]
        hidden = chat_occluded(slots)
        sig = np.array([slot_signature(s) for s in slots])
        changes.append([signature_distance(a, b) for a, b in zip(signatures[-1], sig)] if signatures else [0]*4)
        signatures.append(sig)
        elixir.append(read_elixir(frame, boxes['elixir_bar_box'], cfg))
        occluded.append(hidden)
        if i and i % int(30*cfg.scan_fps) == 0:
            print(f'  Escaneado {t-start:.0f}/{end-start:.0f}s', flush=True)
    return {'times': times, 'elixir': np.array(elixir), 'signatures': np.array(signatures),
            'occluded': np.array(occluded), 'slot_changes': np.array(changes), 'boxes': boxes}


def detect_events(trace, cfg=CFG):
    times, energy = trace['times'], trace['elixir']
    events, blocked = [], []
    # Agrupa únicamente escalones negativos contiguos; no aplica cooldown global de 1.5s.
    negative = np.isfinite(energy[:-1]) & np.isfinite(energy[1:]) & (np.diff(energy) < -0.12)
    indices = np.flatnonzero(negative) + 1
    # Una pausa de un frame dentro de la animación no divide el gasto.
    runs = np.split(indices, np.where(np.diff(indices) > 2)[0]+1) if len(indices) else []
    for run in runs:
        a, b = int(run[0]-1), int(run[-1])
        drop = float(energy[a] - energy[b])
        blocked.append((float(times[a]), float(times[b])))  # También candidatos rechazados bloquean esperar.
        if times[b]-times[a] > cfg.drop_window+1e-6 or drop < cfg.min_elixir_drop:
            continue
        before = max(0, a-1)
        after = min(len(times)-1, b+int(.4*cfg.scan_fps))
        if trace['occluded'][before:after+1].any():
            continue
        base = trace['signatures'][before]
        distances = np.array([[signature_distance(x, y) for x, y in zip(base, s)]
                              for s in trace['signatures'][b:after+1]])
        # Mediana: una elevación/flash aislado no basta; debe haber cambio persistente.
        rotation = np.median(distances, axis=0)
        changed = np.flatnonzero(rotation >= cfg.rotation_threshold)
        if not len(changed):
            continue
        if len(changed) > 1:
            # Un descenso sin meseta con dos rotaciones no permite asignar costes.
            events.append({'t_drop': float(times[a+1]), 'elixir_drop': drop,
                           'hand_slot': None, 'status': 'ambiguous_combo'})
            continue
        events.append({'t_drop': float(times[a+1]), 'elixir_drop': drop,
                       'hand_slot': int(changed[0])+1, 'status': 'detected'})
    # Evita duplicar una caída animada que presenta una pausa de un frame, SOLO
    # si el slot original todavía no se repuso y no se observa una segunda rotación.
    merged = []
    for event in events:
        if merged and event['hand_slot'] is not None and event['hand_slot'] == merged[-1]['hand_slot'] and event['t_drop']-merged[-1]['t_drop'] <= cfg.drop_window:
            merged[-1]['elixir_drop'] += event['elixir_drop']
        else:
            merged.append(event)
    return merged, blocked


def plot_temporal(trace, events):
    fig, axes = plt.subplots(2, 1, figsize=(16, 6), sharex=True)
    axes[0].plot(trace['times'], trace['elixir'], label='Elixir continuo')
    for e in events:
        axes[0].axvline(e['t_drop'], color='crimson', alpha=.6)
    for slot in range(4):
        axes[1].plot(trace['times'], trace['slot_changes'][:,slot], label=f'Slot {slot+1}', alpha=.7)
    axes[0].fill_between(trace['times'], 0, 10, where=trace['occluded'], alpha=.2, color='blue', label='Chat')
    axes[0].set(ylim=(0, 10.5), ylabel='Elixir', title='Fase 1: caídas de elixir y rotación de mano')
    axes[1].set(xlabel='Segundo absoluto del video', ylabel='Cambio visual')
    for ax in axes:
        ax.legend(); ax.grid(alpha=.2)
    plt.tight_layout(); plt.show()


## Subsistema 2 · Banco canónico e identificación
Catálogo público `RoyaleAPI/cr-api-data` (`docs/json/cards_i18n_evo.json`) e ilustraciones de cartas de `RoyaleAPI/cr-api-assets` (`cards/{key}.png`). Se conservan nombres españoles, costes y variantes evolucionadas. El número de cartas depende de la versión disponible del banco; no se fija a 110. El snapshot público verificado contiene 128 plantillas (8 evolucionadas) y corresponde a datos de 2023; no es un catálogo exhaustivo de todas las cartas actuales. Para videos con ilustraciones nuevas o evoluciones ausentes, ampliar `CUSTOM_CATALOG` con referencias verificadas antes de exportar esas cartas. Las identidades de baja confianza se rechazan.

Un error de descarga detiene la construcción del banco y deja un reporte: no se continúa con un banco incompleto. `CUSTOM_CATALOG` permite referencias locales verificadas. Comparación de plantillas e histogramas con búsqueda vertical; primero identidad visual, después validación del gasto. Revisar el piloto para ajustar umbrales: similitud visual no garantiza la identidad.


In [ ]:
def build_catalog():
    manifest = CATALOG_DIR / 'cards_i18n_evo.json'
    if CUSTOM_CATALOG:
        records = json.loads(Path(CUSTOM_CATALOG).read_text())
        source_dir = Path(CUSTOM_CATALOG).parent
    else:
        if not manifest.exists():
            response = requests.get(CATALOG_URL, timeout=60)
            response.raise_for_status()
            atomic_json(manifest, response.json())
        records = json.loads(manifest.read_text())
        source_dir = CATALOG_DIR
    bank, failures = [], []
    for row in records:
        key = row['key']
        try:
            path = source_dir / row['image_path'] if 'image_path' in row else CATALOG_DIR / f'{key}.png'
            if not path.exists():
                response = requests.get(ICON_URL.format(key=key), timeout=30)
                response.raise_for_status()
                atomic_bytes(path, response.content)
            icon = cv2.imread(str(path), cv2.IMREAD_COLOR)
            if icon is None:
                raise ValueError('Icono no decodificable')
            name = row.get('name_es') or row.get('_lang', {}).get('name', {}).get('es')
            if not name:
                raise ValueError('Falta nombre español')
            evolved = bool(row.get('is_evolved', False))
            if evolved and 'evolucionad' not in name.lower():
                name += ' evolucionada'
            # Plantilla interior: sin marco/coste. Mantiene información de variante.
            h, w = icon.shape[:2]
            art = icon[int(.16*h):int(.78*h), int(.12*w):int(.88*w)]
            bank.append({'key': key, 'name': name, 'cost': row['elixir'], 'is_evolved': evolved,
                         'image_path': str(path), 'template': cv2.resize(art, (48, 64))})
        except Exception as exc:
            failures.append({'key': key, 'error': str(exc)})
    atomic_json(CATALOG_DIR / 'download_report.json', {'loaded': len(bank), 'failed': failures})
    if failures:
        raise RuntimeError(f'Banco incompleto: {len(failures)} iconos fallidos. Revisa download_report.json y vuelve a ejecutar.')
    if not bank:
        raise ValueError('Banco vacío')
    print(f'Banco: {len(bank)} plantillas; {sum(c["is_evolved"] for c in bank)} evolucionadas')
    return bank


def classify_card(slot_image, event, bank, cfg=CFG):
    # Búsqueda con tolerancia a elevación: ventana de arte desplazada verticalmente.
    h, w = slot_image.shape[:2]
    candidates = []
    for row in bank:
        template = row['template']
        scores = []
        for top in (.10, .17, .23, .29):
            patch = cv2.resize(slot_image[int(top*h):int((top+.50)*h), int(.15*w):int(.85*w)], (48, 64))
            # La ilustración puede estar en gris si aún no alcanza el elixir.
            gray_patch = cv2.cvtColor(patch, cv2.COLOR_BGR2GRAY)
            gray_template = cv2.cvtColor(template, cv2.COLOR_BGR2GRAY)
            corr = float(cv2.matchTemplate(gray_patch, gray_template, cv2.TM_CCOEFF_NORMED)[0,0])
            def histogram(im):
                hsv = cv2.cvtColor(im, cv2.COLOR_BGR2HSV)
                hist = cv2.calcHist([hsv], [0,1], None, [18,8], [0,180,0,256])
                return cv2.normalize(hist, None)
            similarity = 1 - cv2.compareHist(histogram(patch), histogram(template), cv2.HISTCMP_BHATTACHARYYA)
            saturation = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)[:,:,1].mean()
            scores.append(max(0, corr) if saturation < 25 else .7*max(0, corr) + .3*similarity)
        candidates.append((max(scores), row))
    candidates.sort(key=lambda pair: pair[0], reverse=True)
    score, best = candidates[0]
    # No forzar la identificación por coste: primero visual, después validación.
    margin = score - candidates[1][0] if len(candidates)>1 else score
    cost = best['cost']
    valid_cost = isinstance(cost, (int,float)) and cost >= 1 and abs(cost-event['elixir_drop']) <= cfg.cost_tolerance
    accepted = score >= cfg.card_score_min and margin >= cfg.card_margin_min and valid_cost
    return {'card_name': best['name'], 'card_key': best['key'], 'elixir_cost': cost,
            'card_score': float(score), 'card_margin': float(margin),
            'card_valid': bool(accepted), 'is_evolved': best['is_evolved']}


## Subsistema 3 · Localización y seis zonas
Mapa de diferencias `|I(t+0.3)-I(t−0.3)|`, limitado a la mitad propia y con supresión de movimiento previo. Centroide del componente principal y coordenadas normalizadas sobre **toda** `arena_box`, no sobre la mitad propia. Dos componentes de tamaño similar generan revisión.

Las fronteras de zonas son parámetros geométricos preliminares, no casillas exactas. La galería permite detectar cambios causados por otras tropas. Este método heurístico necesita validación visual; hechizos o despliegues en el campo rival no están representados por estas seis zonas.


In [ ]:
ZONES = ('Detrás del puente izquierdo', 'Detrás del puente derecho',
         'Frente a la torre izquierda', 'Frente a la torre derecha',
         'Centro del campo propio', 'Detrás de la torre del rey')


def placement_zone(coord):
    x, y = coord
    if not (0 <= x <= 1 and .5 <= y <= 1):
        raise ValueError('La especificación de seis zonas solo cubre el campo propio')
    if y >= .87:
        return ZONES[5]
    if .36 <= x <= .64:
        return ZONES[4]
    if y < .65:
        return ZONES[0 if x < .5 else 1]
    return ZONES[2 if x < .5 else 3]


def locate_spawn(reader, event, boxes):
    t = event['t_drop']
    before = crop(reader.at(t-.3), boxes['arena_box'])
    after = crop(reader.at(t+.3), boxes['arena_box'])
    gray_before = cv2.cvtColor(before, cv2.COLOR_BGR2GRAY)
    gray_after = cv2.cvtColor(after, cv2.COLOR_BGR2GRAY)
    diff = cv2.absdiff(gray_before, gray_after)
    # Quita movimiento ya presente antes del despliegue para reducir falsos centros.
    earlier = cv2.cvtColor(crop(reader.at(t-.5), boxes['arena_box']), cv2.COLOR_BGR2GRAY)
    previous_motion = cv2.absdiff(gray_before, earlier)
    mask = ((diff > 30) & (previous_motion < 35)).astype(np.uint8)*255
    h, w = mask.shape
    mask[:int(.5*h)] = 0
    kernel = np.ones((max(3, round(w*.009)),)*2, np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    n, labels, stats, centers = cv2.connectedComponentsWithStats(mask)
    candidates = []
    for i in range(1, n):
        area = int(stats[i, cv2.CC_STAT_AREA])
        if not .00015*w*h <= area <= .08*w*h:
            continue
        x, y = centers[i]
        candidates.append((area, [float(x/w), float(y/h)]))
    candidates.sort(reverse=True)
    if not candidates:
        return {'placement_coord': None, 'placement_zone': None, 'spatial_valid': False}, diff
    area, coord = candidates[0]
    # Dos cambios comparables: registrar ambigüedad en vez de inventar la posición.
    ambiguous = len(candidates)>1 and candidates[1][0] > .8*area
    return {'placement_coord': coord, 'placement_zone': placement_zone(coord),
            'spatial_valid': not ambiguous, 'spatial_candidates': len(candidates)}, diff


def label_events(reader, trace, events, bank, match, cfg=CFG):
    labeled = []
    for original in events:
        event = dict(original)
        t = event['t_drop']
        if event['hand_slot'] is None:
            labeled.append(event); continue
        if t-cfg.pre_decision-6 < match['start_sec'] or t+.4 > match['end_sec']:
            event['status'] = 'insufficient_context'
            labeled.append(event); continue
        pre = reader.at(t-cfg.card_lookback)
        slots = [crop(pre, b) for b in trace['boxes']['card_slots']]
        if chat_occluded(slots):
            event['status'] = 'occluded'
            labeled.append(event); continue
        event.update(classify_card(slots[event['hand_slot']-1], event, bank, cfg))
        event.update(locate_spawn(reader, event, trace['boxes'])[0])
        event['status'] = 'accepted' if event['card_valid'] and event['spatial_valid'] else 'needs_review'
        labeled.append(event)
    return labeled


def event_gallery(reader, trace, events, bank=None, limit=12):
    for event in events[:limit]:
        t = event['t_drop']
        fig, axes = plt.subplots(1, 3, figsize=(12, 5))
        frame = reader.at(t-CFG.card_lookback)
        slot = event.get('hand_slot')
        axes[0].imshow(cv2.cvtColor(crop(frame, trace['boxes']['card_slots'][slot-1]) if slot else frame, cv2.COLOR_BGR2RGB))
        axes[0].set_title(f'Slot previo {slot}\nt={t-.8:.2f}s')
        spatial, diff = locate_spawn(reader, event, trace['boxes'])
        axes[1].imshow(diff, cmap='hot'); axes[1].set_title('Diferencias t±0.3s')
        arena = crop(reader.at(t+.3), trace['boxes']['arena_box'])
        axes[2].imshow(cv2.cvtColor(arena, cv2.COLOR_BGR2RGB))
        coord = event.get('placement_coord') or spatial['placement_coord']
        if coord:
            axes[2].plot(coord[0]*arena.shape[1], coord[1]*arena.shape[0], 'r+', markersize=22)
        axes[2].set_title(f"{event.get('card_name', 'Sin clasificar')}\n{event.get('placement_zone', '')}")
        for ax in axes: ax.axis('off')
        fig.suptitle(f"t_drop={t:.2f}s · ΔE={event['elixir_drop']:.2f} · {event['status']}")
        plt.tight_layout(); plt.show()


## Subsistema 4 · Muestras y exportación
Para cada jugada aceptada: `t_actual=t_drop−0.65`, cuatro frames en `t_actual−6, −4, −2, 0`. El slot se identifica aparte en `t_drop−0.8`. Nunca se entregan frames posteriores al despliegue al VLM. No se rellenan los primeros seis segundos repitiendo imágenes.

`match_index` es cero-based en el dataset y `m1` es la primera partida en los nombres. `timestamp_sec` es el segundo absoluto de decisión; `t_drop_sec` registra el despliegue aparte. `hand_slot` toma valores 1–4.

Espera: ventanas de tres segundos sin eventos ni señales sospechosas. Objetivo 17.5% (intervalo 15–20%); si faltan ventanas válidas se informa, sin inventar muestras. La ausencia de una jugada es una etiqueta observada, no una demostración de optimalidad estratégica.

Cada video guarda `metadata/actions/<id>.jsonl` mediante reemplazo atómico, un estado con huellas de entradas/configuración y `review/<id>.json`. Se suben primero los JPEG y al final el JSONL. Un fallo deja el checkpoint local disponible para reintentar. Al cambiar `UPLOAD_TO_R2=True`, se pueden subir los checkpoints locales sin reprocesar. El consolidado `metadata/actions.jsonl` se genera localmente; los checkpoints por video son la persistencia R2 requerida.


In [ ]:
def wait_times(trace, events, blocked, match, n, cfg=CFG):
    candidates = []
    for t in np.arange(float(match['start_sec'])+6, float(match['end_sec'])-cfg.wait_stability+1e-6, cfg.wait_stability):
        # Ventana futura completa de 3s sin jugadas, incluidos candidatos rechazados.
        if any(t <= e['t_drop'] <= t+cfg.wait_stability for e in events):
            continue
        if any(a <= t+cfg.wait_stability and b >= t for a,b in blocked):
            continue
        indices = (trace['times'] >= t) & (trace['times'] <= t+cfg.wait_stability)
        if not indices.any() or trace['occluded'][indices].any() or not np.isfinite(trace['elixir'][indices]).all():
            continue
        # Rotaciones sin caída suficiente (p.ej. cartas de 1) no deben etiquetarse esperar.
        if trace['slot_changes'][indices].max() >= cfg.rotation_threshold:
            continue
        history = [np.argmin(abs(trace['times']-(t-d))) for d in (6,4,2,0)]
        if trace['occluded'][history].any():
            continue
        candidates.append(float(t))
    rng = np.random.default_rng(cfg.seed)
    return sorted(rng.choice(candidates, size=min(n,len(candidates)), replace=False).tolist()) if candidates else []


def export_sample(reader, video_id, match_index, sample_index, t, match, layout, event=None, cfg=CFG):
    action = 'play' if event else 'wait'
    sid = f'{video_id}_m{match_index+1}_{action}_{sample_index:03d}'
    times = [t-offset for offset in (6,4,2,0)]
    if times[0] < match['start_sec'] or times[-1] > match['end_sec']:
        raise ValueError('Contexto fuera de la partida; no se duplican frames iniciales')
    if event and not .5-1e-6 <= event['t_drop']-t <= .8+1e-6:
        raise ValueError('El frame de decisión incumple prevención de leakage')
    frames = []
    for sec, offset in zip(times, (6,4,2,0)):
        frame = reader.at(sec)
        boxes = boxes_for(layout, frame)
        if chat_occluded([crop(frame,b) for b in boxes['card_slots']]):
            return None
        key = f'samples/{sid}_t-{offset}s.jpg'
        ok, encoded = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
        if not ok:
            raise IOError('Fallo codificando JPEG')
        atomic_bytes(OUTPUT_DIR/key, encoded.tobytes())
        frames.append(key)
    row = {'sample_id': sid, 'video_id': video_id, 'match_index': match_index,
           'timestamp_sec': round(t,6), 'frames': frames, 'action_type': action,
           'card_name': None, 'elixir_cost': None, 'hand_slot': None,
           'placement_coord': None, 'placement_zone': None, 'target_text': 'esperar',
           'frame_timestamps_sec': times}
    if event:
        for key in ('card_name','elixir_cost','hand_slot','placement_coord','placement_zone'):
            row[key] = event[key]
        row['target_text'] = f"Juega {row['card_name']} {row['placement_zone'].lower()}"
        row['t_drop_sec'] = event['t_drop']
    return row


def fingerprint(video_info, layout, bank, cfg=CFG):
    cards = [{k:v for k,v in c.items() if k not in ('template','image_path')} | {
        'template_sha256': hashlib.sha256(c['template'].tobytes()).hexdigest()} for c in bank]
    payload = {'version': '4B-1', 'video': video_info, 'layout': layout, 'config': asdict(cfg), 'cards': cards}
    return hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()


def publish_video(video_id, rows, client=None):
    # Toda escritura remota pasa por esta función. Comprobar el flag en tiempo de llamada.
    if not UPLOAD_TO_R2:
        return False
    if client is None:
        raise RuntimeError('UPLOAD_TO_R2=True pero faltan credenciales')
    # Imágenes primero; JSONL al final funciona como commit del video.
    for row in rows:
        for key in row['frames']:
            client.upload_file(str(OUTPUT_DIR/key), R2_BUCKET, key, ExtraArgs={'ContentType':'image/jpeg'})
    key = f'metadata/actions/{video_id}.jsonl'
    client.put_object(Bucket=R2_BUCKET, Key=key, Body=(OUTPUT_DIR/key).read_bytes(), ContentType='application/x-ndjson')
    return True


def process_video(video_id, video_info, layout, bank, client=None, cfg=CFG):
    checkpoint = OUTPUT_DIR / f'metadata/actions/{video_id}.jsonl'
    state_path = OUTPUT_DIR / f'metadata/actions/{video_id}.state.json'
    fp = fingerprint(video_info, layout, bank, cfg)
    if SKIP_COMPLETED and checkpoint.exists() and state_path.exists():
        state = json.loads(state_path.read_text())
        if state.get('fingerprint') == fp and state.get('sha256') == hashlib.sha256(checkpoint.read_bytes()).hexdigest():
            rows = [json.loads(line) for line in checkpoint.read_text().splitlines()]
            if all((OUTPUT_DIR/f).exists() for row in rows for f in row['frames']):
                publish_video(video_id, rows, client)  # Permite subir después de una ejecución local.
                print(f'{video_id}: checkpoint válido, {len(rows)} muestras', flush=True)
                return rows
    path, downloaded = fetch_video(video_id)
    reader = VideoReader(path)
    rows, review, stats = [], [], []
    try:
        for match_index, match in enumerate(video_info['matches']):
            print(f'{video_id} · partida {match_index+1}', flush=True)
            trace = scan_match(reader, match, layout, cfg)
            events, blocked = detect_events(trace, cfg)
            labeled = label_events(reader, trace, events, bank, match, cfg)
            review.extend(dict(e, match_index=match_index) for e in labeled)
            plays = []
            for i, event in enumerate(labeled, 1):
                if event['status'] == 'accepted':
                    sample = export_sample(reader, video_id, match_index, i, event['t_drop']-cfg.pre_decision, match, layout, event, cfg)
                    if sample: plays.append(sample)
            n_wait = round(len(plays)*cfg.wait_fraction/(1-cfg.wait_fraction))
            passive = []
            for i, t in enumerate(wait_times(trace, events, blocked, match, n_wait, cfg),1):
                sample = export_sample(reader, video_id, match_index, i, t, match, layout, cfg=cfg)
                if sample: passive.append(sample)
            rows.extend(plays+passive)
            stats.append({'match_index':match_index,'detected':len(events),'play':len(plays),'wait':len(passive),
                          'wait_fraction':len(passive)/max(1,len(plays)+len(passive))})
        atomic_json(OUTPUT_DIR/f'review/{video_id}.json', {'events':review,'matches':stats})
        content = ''.join(json.dumps(r, ensure_ascii=False, allow_nan=False)+'\n' for r in rows).encode()
        atomic_bytes(checkpoint, content)
        atomic_json(state_path, {'fingerprint':fp,'sha256':hashlib.sha256(content).hexdigest(), 'samples':len(rows)})
        publish_video(video_id, rows, client)
        print(f'{video_id}: {len(rows)} muestras; revisión: review/{video_id}.json', flush=True)
        return rows
    finally:
        reader.close()
        if downloaded and DELETE_DOWNLOADED_VIDEOS:
            path.unlink(missing_ok=True)


def consolidate():
    rows = []
    for path in sorted((OUTPUT_DIR/'metadata/actions').glob('*.jsonl')):
        state_path = path.with_suffix('.state.json')
        if not state_path.exists():
            continue
        state = json.loads(state_path.read_text())
        if state.get('sha256') != hashlib.sha256(path.read_bytes()).hexdigest():
            continue
        rows.extend(json.loads(line) for line in path.read_text().splitlines())
    target = OUTPUT_DIR/'metadata/actions.jsonl'
    atomic_bytes(target, ''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in rows).encode())
    waits = sum(r['action_type']=='wait' for r in rows)
    print(f'Consolidado local: {len(rows)} muestras · esperar: {waits/max(1,len(rows)):.1%}')
    if rows and not .15 <= waits/len(rows) <= .20:
        print('Balance fuera de 15–20%: revisar ventanas disponibles; no se fabricaron ejemplos.')
    return target


## Cargar entradas y elegir el video piloto
Las lecturas remotas de manifiestos usan S3 directamente. Si falla la lectura, se informa el error en vez de usar silenciosamente datos locales desactualizados.

In [ ]:
R2_BUCKET = get_config('R2_BUCKET', 'cr-videos')
s3_client = make_s3() if (READ_FROM_R2 or UPLOAD_TO_R2) else None
segmented_manifest = read_manifest('segmented_matches.json', s3_client)
layout_manifest = read_manifest('ui_layouts.json', s3_client)
videos = segmented_manifest['videos']
layouts = layout_manifest.get('layouts', layout_manifest)
available = [vid for vid in videos if vid in layouts and videos[vid].get('matches')]
if not available:
    raise ValueError('No hay videos con segmentación y layout compatibles')
if VIDEO_FILTER == 'all':
    target_ids = available
elif isinstance(VIDEO_FILTER, int):
    target_ids = available[:VIDEO_FILTER]
else:
    requested = [VIDEO_FILTER] if isinstance(VIDEO_FILTER,str) else list(VIDEO_FILTER)
    missing = set(requested)-set(available)
    if missing: raise ValueError(f'Videos sin segmentación/layout: {missing}')
    target_ids = requested
pilot_id = PILOT_VIDEO_ID or available[0]
pilot_match = videos[pilot_id]['matches'][PILOT_MATCH_INDEX]
pilot_path, pilot_downloaded = fetch_video(pilot_id)
pilot_reader = VideoReader(pilot_path)
print(f'Piloto: {pilot_id}, partida {PILOT_MATCH_INDEX+1}; lote: {len(target_ids)} videos')


## Fase 1 · Curva de elixir y rotaciones del piloto
Revisa la curva, las líneas de eventos y la ROI amarilla del relleno. Si esta última no coincide con el relleno de la barra, ajusta `CFG.elixir_fill_roi` y repite desde esta fase.

In [ ]:
pilot_trace = scan_match(pilot_reader, pilot_match, layouts[pilot_id])
pilot_events, pilot_blocked = detect_events(pilot_trace)
plot_temporal(pilot_trace, pilot_events)
print(f'{len(pilot_events)} candidatos. Umbral fijo del documento: caída ≥1.5 en ≤0.5s.')
# Validación del recorte: la línea amarilla encierra SOLO el relleno continuo.
frame = pilot_reader.at(pilot_match['start_sec']+8)
elixir_preview = crop(frame, pilot_trace['boxes']['elixir_bar_box']).copy()
h,w = elixir_preview.shape[:2]
x,y,rw,rh = CFG.elixir_fill_roi
cv2.rectangle(elixir_preview,(round(x*w),round(y*h)),(round((x+rw)*w),round((y+rh)*h)),(0,255,255),1)
plt.figure(figsize=(14,3)); plt.imshow(cv2.cvtColor(elixir_preview,cv2.COLOR_BGR2RGB)); plt.title('ROI de elixir: revisar antes de continuar'); plt.axis('off'); plt.show()


## Fase 2 · Construir catálogo e identificar las cartas
El análisis devuelve también confianza visual, margen frente al segundo candidato y consistencia del coste.

In [ ]:
card_bank = build_catalog()
# Identificación del slot exactamente en t_drop −0.8s, seguida de validación de coste.
pilot_labeled = label_events(pilot_reader, pilot_trace, pilot_events, card_bank, pilot_match)
print('Aceptados:', sum(e['status']=='accepted' for e in pilot_labeled), '/', len(pilot_labeled))


## Fase 3 · Galería de localización
Contrasta carta previa, mapa de diferencias y cruz de despliegue. `needs_review` no se exporta automáticamente como etiqueta de entrenamiento.

In [ ]:
event_gallery(pilot_reader, pilot_trace, pilot_labeled, card_bank, limit=12)
# El heatmap y el frame t+0.3 son diagnósticos; nunca forman parte de la entrada VLM.


## Fase 4 · Exportación masiva y checkpointing por video
Activa `RUN_BATCH=True` y ejecuta esta celda. Con `UPLOAD_TO_R2=False`, todo queda local. Al activar subidas, se usa el bucket `cr-videos` por defecto. Reejecutar el lote reanuda checkpoints válidos y reintenta errores.

In [ ]:
if RUN_BATCH:
    pilot_reader.close()
    errors = {}
    for video_id in target_ids:
        try:
            process_video(video_id, videos[video_id], layouts[video_id], card_bank, s3_client)
        except Exception as exc:
            errors[video_id] = f'{type(exc).__name__}: {exc}'
            print(f'ERROR {video_id}: {errors[video_id]}', flush=True)
    atomic_json(OUTPUT_DIR/'review/batch_errors.json', errors)
    consolidated_path = consolidate()
    print('Salida:', OUTPUT_DIR.resolve())
    if errors:
        raise RuntimeError(f'{len(errors)} videos fallaron; consultar review/batch_errors.json y reanudar.')
else:
    print('Piloto listo. Tras revisar la galería, cambia RUN_BATCH=True y ejecuta esta celda.')


## Liberar lector del piloto

In [ ]:
# Ejecutar al terminar la revisión si no vas a lanzar el lote en esta sesión.
pilot_reader.close()
